In [ ]:
%pip -q install seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load datasets
train_path = "C:\\Users\\Madis\\Downloads\\preprocessing\\train_cleaned.csv"
val_path = "C:\\Users\\Madis\\Downloads\\preprocessing\\validation_cleaned.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)

Training dataset: (19300, 530)
Validation dataset: (1111, 530)


In [4]:
# Define the Zero-Rule Baseline Classifier directly
class ZeroRuleClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.classes_, counts = np.unique(y, return_counts=True)
        self.majority_class_ = self.classes_[np.argmax(counts)]
        return self
    def predict(self, X):
        return np.array([self.majority_class_] * len(X))
    def score(self, X, y):
        preds = self.predict(X)
        return np.mean(preds == y)



In [5]:
# Identify Wi-Fi RSSI features
wap_cols = [
    col for col in train_df.columns
    if col.startswith("WAP")
]

# Separate features and target
X_raw = train_df[wap_cols].copy()
y = train_df["LOCATION_CLASS"].copy()

X_validation_raw = val_df[wap_cols].copy()
y_validation = val_df["LOCATION_CLASS"].copy()

print("Number of WAP features:", len(wap_cols))
print("Number of location classes:", y.nunique())

# Check for conventional missing values
print("NaN values in training:", X_raw.isna().sum().sum())
print("NaN values in validation:", X_validation_raw.isna().sum().sum())

Number of WAP features: 520
Number of location classes: 13
NaN values in training: 0
NaN values in validation: 0


In [6]:
# CONFIGURATION A: Original RSSI encoding
X_original = X_raw.copy()
X_validation_original = X_validation_raw.copy()

# CONFIGURATION B: Replace undetected WAP values (100) with -100
X_minus110 = X_raw.replace(100, -110)
X_validation_minus110 = X_validation_raw.replace(100, -110)

# Store both configurations
configurations = {
    "Original (100)": (X_original, X_validation_original),
    "Modified (-110)": (X_minus110, X_validation_minus110)
}

# Verify replacement
print("Original encoding - number of 100 values:",
      (X_original == 100).sum().sum())

print("Modified encoding - number of 100 values:",
      (X_minus110 == 100).sum().sum())

print("Modified encoding - number of -110 values:",
      (X_minus110 == -110).sum().sum())

Original encoding - number of 100 values: 0
Modified encoding - number of 100 values: 0
Modified encoding - number of -110 values: 9690920


In [8]:
# Create sample indices
indices = np.arange(len(X_raw))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split target labels
y_train = y.iloc[train_indices]
y_test = y.iloc[test_indices]

print("Training samples:", len(train_indices))
print("Testing samples:", len(test_indices))

Training samples: 15440
Testing samples: 3860


In [9]:
from sklearn.metrics import accuracy_score, f1_score

# Store results
zero_rule_results = []

# Evaluate both RSSI configurations
for config_name, (X_config, X_val_config) in configurations.items():

    # Split features using the same indices
    X_train = X_config.iloc[train_indices]
    X_test = X_config.iloc[test_indices]

    # Initialize Zero-Rule classifier
    baseline = ZeroRuleClassifier()

    # Train classifier
    baseline.fit(X_train, y_train)

    # Generate predictions
    preds = baseline.predict(X_test)

    # Calculate metrics
    baseline_acc = accuracy_score(y_test, preds)
    baseline_f1 = f1_score(
        y_test,
        preds,
        average='macro'
    )

    # Store results
    zero_rule_results.append({
        "Configuration": config_name,
        "Accuracy": baseline_acc,
        "Macro-F1": baseline_f1
    })

    print(f"\nZero-Rule Baseline: {config_name}")
    print(f"Test Accuracy: {baseline_acc * 100:.2f}%")
    print(f"Macro-F1: {baseline_f1:.4f}")


# Display comparison table
zero_rule_comparison = pd.DataFrame(zero_rule_results)

print("\nZERO-RULE CONFIGURATION COMPARISON")
display(zero_rule_comparison.round(4))


Zero-Rule Baseline: Original (100)
Test Accuracy: 14.04%
Macro-F1: 0.0189

Zero-Rule Baseline: Modified (-110)
Test Accuracy: 14.04%
Macro-F1: 0.0189

ZERO-RULE CONFIGURATION COMPARISON


,Configuration,Accuracy,Macro-F1
0,Original (100),0.1404,0.0189
1,Modified (-110),0.1404,0.0189


In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

In [13]:
import numpy as np

class KNearestNeighbor(object):
    """A kNN classifier with L2 distance."""

    def __init__(self):
        pass

    def train(self, X, y):
        """
        Train the classifier by memorizing the training data.
        """
        self.X_train = X
        self.y_train = y

    def predict(self, X, k=5):
        """
        Predict labels for test data using k nearest neighbors.
        """
        dists = self.compute_distances(X)

        return self.predict_labels(dists, k=k)

    def compute_distances(self, X):
        """
        Compute Euclidean distance between test and training samples.
        """
        '''Calculate squared distances'''
        dists = (
        np.sum(X**2, axis=1, keepdims=True)
        + np.sum(self.X_train**2, axis=1)
        - 2 * np.dot(X, self.X_train.T)
    )

        # Prevent small negative values caused by floating-point errors
        dists = np.maximum(dists, 0)

        # Take square root to get Euclidean distances
        dists = np.sqrt(dists)
        return dists

    def predict_labels(self, dists, k=5):
        """
        Predict labels using majority voting among k nearest neighbors.
        """
        num_test = dists.shape[0]
        y_pred = np.empty(num_test, dtype=self.y_train.dtype)

        for i in range(num_test):

            # Find indices of the k closest training samples
            kmin_idx = np.argsort(dists[i, :])[:k]

            # Get labels of the closest neighbors
            closest_y = self.y_train[kmin_idx]

            # Find the most common label
            idx, counts = np.unique(closest_y, return_counts=True)

            y_pred[i] = idx[np.argmax(counts)]

        return y_pred

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

knn_results = []

for config_name, (X_config, X_val_config) in configurations.items():

    # Separate training and testing data
    X_train = X_config.iloc[train_indices].to_numpy()
    X_test = X_config.iloc[test_indices].to_numpy()

    # Convert labels to NumPy arrays
    y_train_array = y_train.to_numpy()
    y_test_array = y_test.to_numpy()

    # Scale features
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Initialize and train KNN
    knn = KNearestNeighbor()
    knn.train(X_train_scaled, y_train_array)

    # Predict using 5 nearest neighbors
    predictions = knn.predict(X_test_scaled, k=5)

    # Calculate performance
    accuracy = accuracy_score(y_test_array, predictions)
    macro_f1 = f1_score(
        y_test_array,
        predictions,
        average="macro"
    )

    # Save results
    knn_results.append({
        "Configuration": config_name,
        "Accuracy": accuracy,
        "Macro-F1": macro_f1
    })

    print(f"\nKNN Results: {config_name}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%")
    print(f"Macro-F1: {macro_f1:.4f}")


# Display comparison
knn_comparison = pd.DataFrame(knn_results)

print("\nKNN CONFIGURATION COMPARISON")
display(knn_comparison.round(4))


KNN Results: Original (100)
Test Accuracy: 98.08%
Macro-F1: 0.9804

KNN Results: Modified (-110)
Test Accuracy: 98.08%
Macro-F1: 0.9804

KNN CONFIGURATION COMPARISON


,Configuration,Accuracy,Macro-F1
0,Original (100),0.9808,0.9804
1,Modified (-110),0.9808,0.9804
